In [1]:
pip install transformers torch joblib scikit-learn datasets evaluate accelerate

  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached torch-2.6.0-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached datasets-3.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.3-py3-none-any.whl.metadata (9.2 kB)
  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached huggingface_hub-0.30.2-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached regex-2024.11.6-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.5.3-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.8 kB)
  Using cached networkx-3.4.2-p

In [12]:
import torch
from transformers import AutoModelForTokenClassification
from torch.nn.utils import parameters_to_vector

In [13]:
base_model = AutoModelForTokenClassification.from_pretrained("base_model")

taskB_model = AutoModelForTokenClassification.from_pretrained("model_taskB_ner")

In [14]:
for param in base_model.parameters():
    param.requires_grad = False

for param in taskB_model.parameters():
    param.requires_grad = False

In [15]:
theta_0 = parameters_to_vector(base_model.parameters()).detach()  # θ₀
theta_B = parameters_to_vector(taskB_model.parameters()).detach()  # θB

In [16]:
if theta_0.size() != theta_B.size():
    min_size = min(theta_0.size(0), theta_B.size(0))
    theta_0 = theta_0[:min_size]
    theta_B = theta_B[:min_size]

In [17]:
delta_B = theta_B - theta_0

In [18]:
torch.save(delta_B, "delta_taskB_ner.pt")

# Merging the Task Vectors (ΔA and ΔB)

In [20]:
delta_A = torch.load("delta_taskA_sst2.pt")
delta_B = torch.load("delta_taskB_ner.pt")

In [22]:
min_size = min(delta_A.size(0), delta_B.size(0))  # Get the smaller size

# Truncate the larger tensor to match the smaller one
delta_A = delta_A[:min_size]
delta_B = delta_B[:min_size]

In [23]:
delta_merged = (delta_A + delta_B) / 2

In [ ]:
# weight_A = 0.7
# weight_B = 0.3
# delta_merged = weight_A * delta_A + weight_B * delta_B

In [24]:
torch.save(delta_merged, "delta_merged.pt")